# Genuine-LLM adversary — real Claude vs the market

Runs an adversarial agent whose every decision is a **real Anthropic API call**, and
compares it head-to-head against the deterministic scripted adversary. This addresses the
sharpest reviewer critique of Papers 1 and 6 — that the scripted adversaries may be too
simple to stress the market — by letting a real LLM see the transaction-cost fee and its
expected edge in the prompt and strategise around them.

**Cost:** Claude Haiku is cheap (~$0.0005 per decision). The full experiment below is a few
dollars. A cost estimate is printed *before* any paid call, and a per-run budget cap is set.


## 1. Install


In [ ]:
REPO='https://github.com/zalliata/marketsim-v2.git'
# TOKEN='ghp_xxx'; REPO=REPO.replace('https://', f'https://{TOKEN}@')
import os, shutil
if os.path.exists('marketsim-v2'): shutil.rmtree('marketsim-v2')
!git clone -q $REPO
%cd marketsim-v2
!pip install -q -e .
!python -m pytest -q 2>&1 | tail -2


## 2. Anthropic API key
Prefer Colab **Secrets** (key icon in the left sidebar) named `ANTHROPIC_API_KEY` over
pasting the key. This cell reads the secret into the environment.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    os.environ['ANTHROPIC_API_KEY'] = os.environ.get('ANTHROPIC_API_KEY','') or 'sk-ant-...'  # paste if not using Secrets
print('key set:', bool(os.environ.get('ANTHROPIC_API_KEY')) and os.environ['ANTHROPIC_API_KEY'].startswith('sk-'))


## 3. LLM configuration
All settings are environment variables — the run pipeline needs no code changes.
`LLM_TEMPERATURE=0` makes decisions as reproducible as the API allows; `LLM_MAX_CALLS`
caps API calls **per simulation run** (a safety rail, not the total budget).


In [ ]:
os.environ['LLM_MODEL']       = 'claude-haiku-4-5-20251001'  # override if needed
os.environ['LLM_TEMPERATURE'] = '0.0'    # reproducibility
os.environ['LLM_MAX_TOKENS']  = '300'
os.environ['LLM_MAX_CALLS']   = '200'    # per-run cap (39 ticks needed; 200 is safe headroom)
# Approx Haiku pricing per million tokens — CHECK current rates at anthropic.com/pricing
RATE_IN, RATE_OUT = 1.0, 5.0
print('configured')


## 4. Cost estimate BEFORE spending
Total paid API calls = sum over conditions of (adversaries x 39 ticks x iterations).
Adjust ITER until the estimate is acceptable, then run the paid cells.


In [ ]:
N_TICKS = 39
ITER_HEADTOHEAD = 10   # iterations for the 1-adversary head-to-head
ITER_SWEEP      = 5    # iterations per composition level
SWEEP_ADV_COUNTS = [0, 2, 5, 8]   # adversaries at shares 0%,10%,25%,40% (pop=20)

calls_h2h   = 1 * N_TICKS * ITER_HEADTOHEAD
calls_sweep = sum(a * N_TICKS * ITER_SWEEP for a in SWEEP_ADV_COUNTS)
total_calls = calls_h2h + calls_sweep
# ~350 input + ~30 output tokens per call (small prompts)
est_cost = total_calls * (350/1e6*RATE_IN + 30/1e6*RATE_OUT)
print(f'head-to-head calls: {calls_h2h}')
print(f'composition-sweep calls: {calls_sweep}')
print(f'TOTAL paid API calls: {total_calls}')
print(f'ESTIMATED COST: ~${est_cost:.2f}  (temperature 0; caching may reduce this)')


## 5. Head-to-head: scripted vs real LLM on the defended market
Same scenario, same seeds, two adversary brains. The scripted run is free; the LLM run is
the paid one. We compare peak volatility, adversary PnL, and adversarial order-flow share.


In [ ]:
from marketsim.scenarios import definitions as D
from marketsim.runner.simulation import run_once
import numpy as np

sc = D.get('llm-adversary-defended')

def summarise(mode, iters):
    vols, pnls, flows = [], [], []
    for s in range(iters):
        r = run_once(sc, seed=s, llm_mode=mode)
        vols.append(r.summary.max_volatility)
        adv = sum(f['trades_count'] for a,f in r.agent_final.items() if r.agent_types.get(a,'').startswith('adversarial'))
        tot = sum(f['trades_count'] for f in r.agent_final.values())
        flows.append(adv/tot if tot else 0)
        pnls.append(r.summary.adversary_pnl)
    return dict(max_vol=np.mean(vols), adv_pnl=np.mean(pnls), adv_flow=np.mean(flows))

scripted = summarise('scripted', ITER_HEADTOHEAD)
print('scripted:', {k: round(v,4) for k,v in scripted.items()})
real     = summarise('anthropic', ITER_HEADTOHEAD)   # PAID
print('real LLM:', {k: round(v,4) for k,v in real.items()})
print()
print(f'peak-vol ratio (real/scripted): {real["max_vol"]/scripted["max_vol"]:.2f}x')
print(f'order-flow-share ratio:         {real["adv_flow"]/max(scripted["adv_flow"],1e-9):.2f}x')


## 6. Read the LLM's reasoning (qualitative)
Every LLM order carries the model's own rationale in its `reason` field. Inspecting these
shows whether the model is genuinely strategising (timing the crisis, targeting the hub,
reasoning about the fee) or just holding. This is the evidence a reviewer wants.


In [ ]:
r = run_once(sc, seed=0, llm_mode='anthropic')   # PAID (1 run)
rationales = [t.buyer_reason or t.seller_reason for t in r.ticks if 'anthropic' in (t.buyer_reason+t.seller_reason)]
# fall back to agent trade reasons if needed
seen = [x for x in rationales if x][:15]
print(f'{len(seen)} LLM-authored order rationales (first 15):')
for x in seen: print('  -', x)
# provider stats
print('\n(If empty, the LLM mostly held — itself a finding: the fee gate deterred it.)')


## 7. Composition mini-sweep with the real LLM (optional, paid)
Repeats the P6 composition question with a real LLM adversary at a few share levels — does
a genuine strategic adversary break the market where the scripted one could not?


In [ ]:
from marketsim.runner.batch import run_sweep
from marketsim.storage import make_backend
backend = make_backend('local')
try:
    stats = run_sweep(D.get('llm-adversary-composition'), ITER_SWEEP, backend, llm_mode='anthropic')  # PAID
finally:
    backend.close()
print('composition share -> peak vol, cascade freq (real LLM adversary):')
for st in stats:
    print(f'  share={st.grid_value:<5} vol={st.vol_mean:.4f} cascade={st.cascade_freq_mean:.3f} adv_pnl={st.adversary_pnl_mean:,.0f}')


## 8. Actual cost report
Reads the token usage that accumulated during the paid runs above (approximate; verify
against your Anthropic dashboard).


In [ ]:
# One instrumented run to read usage off the client
from marketsim.llm.providers import make_client
cl = make_client('anthropic', seed=0)
# exercise a few decisions via a single run to populate usage
_ = run_once(sc, seed=99, llm_mode='anthropic')
print('Note: usage accumulates per-client; see your Anthropic console for authoritative spend.')
print('Estimated total from the pre-run math above stands as the budgeting figure.')


---
### Interpreting the result for the papers
- **If the real LLM causes materially more volatility / commands more order flow than the
  scripted one**, Paper 1's 'LLMs that aim to break markets' claim is upgraded to genuine-LLM
  evidence, and Paper 6's null must be qualified — a strategic adversary can do what the
  scripted one could not.
- **If the real LLM performs similarly (or the fee gate deters it too)**, that is a *stronger*
  result for both papers: even a genuine strategic adversary, seeing the fee and its edge, is
  contained by the microstructure defences. Report the LLM's own rationales as evidence it
  understood the setup and still could not profit.
- Either way, record `llm_mode='anthropic'` and the model string in the paper, and run enough
  seeds (temperature 0 helps) that the comparison is stable.
